In [ ]:
from pathlib import Path
import os
import sys
_release_root = Path(os.environ.get('AVATAR_RELEASE_ROOT', Path.cwd())).resolve()
while not (_release_root / 'avatar_paths.py').is_file():
    if _release_root == _release_root.parent:
        raise RuntimeError('Open this notebook from the code package directory.')
    _release_root = _release_root.parent
if str(_release_root) not in sys.path:
    sys.path.insert(0, str(_release_root))
from avatar_paths import project_path, get_value, get_column, get_path, get_label


In [ ]:
from pathlib import Path
import os
import re
import json
import math
import random
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import torch
import torch.nn as nn
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss, confusion_matrix, balanced_accuracy_score, precision_score, recall_score, f1_score
try:
    from lifelines import KaplanMeierFitter, CoxPHFitter
    from lifelines.statistics import logrank_test
    from lifelines.utils import concordance_index
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('This notebook requires lifelines. Install it with: pip install lifelines openpyxl') from exc
from IPython.display import display
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
RANDOM_STATE = 20260802
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



In [ ]:
PROJECT_DIR = Path(str(project_path(get_value('science.paths.directory_008', 'directory_008'))))
INPUT_DIR = PROJECT_DIR / get_value('science.paths.directory_009', 'directory_009')
MODEL_RESULTS_DIR = PROJECT_DIR / get_value('science.paths.directory_010', 'directory_010')
STAGE2_DIR = MODEL_RESULTS_DIR / get_value('science.paths.directory_011', 'directory_011')
STAGE3_DIR = MODEL_RESULTS_DIR / get_value('science.paths.directory_015', 'directory_015')
EXTERNAL_LABEL_PATH = Path(str(project_path(get_value('science.tables.outcomes_011', 'outcomes_011.xlsx'))))
BASE_OUTPUT_DIR = MODEL_RESULTS_DIR / get_value('science.paths.directory_050', 'directory_050')
N_BOOTSTRAP = 1000
BOOTSTRAP_ALPHA = 0.05
STRICT_MODE = True
SURVIVAL_GROUP_LABELS = ['Lower risk', 'Higher risk']
MODALITY_CONFIG = {'Metagenomics': {'external_matrix': INPUT_DIR / get_value('science.tables.features_021', 'features_021.csv.gz'), 'label_modal': 'microbiota', 'display_name': 'Metagenomics (KO)'}, 'Radiomics': {'external_matrix': INPUT_DIR / get_value('science.tables.features_022', 'features_022.csv.gz'), 'label_modal': 'radiomics', 'display_name': 'CT radiomics'}, 'Blood': {'external_matrix': INPUT_DIR / get_value('science.tables.features_023', 'features_023.csv.gz'), 'label_modal': 'blood', 'display_name': 'Blood metabolomics'}, 'Urine': {'external_matrix': INPUT_DIR / get_value('science.tables.features_024', 'features_024.csv.gz'), 'label_modal': 'urine', 'display_name': 'Urine metabolomics'}}
RUN_MODALITY = 'all'
_MODALITY_ALIASES = {'all': 'all', 'metagenomics': 'Metagenomics', 'microbiota': 'Metagenomics', 'microbiome': 'Metagenomics', 'ko': 'Metagenomics', 'radiomics': 'Radiomics', 'radiomic': 'Radiomics', 'ct': 'Radiomics', 'blood': 'Blood', 'urine': 'Urine'}
_run_key = str(RUN_MODALITY).strip().lower()
if _run_key not in _MODALITY_ALIASES:
    raise ValueError(f'RUN_MODALITY={RUN_MODALITY!r} is invalid. Available options: all, Metagenomics, Radiomics, Blood, Urine.')
_selected_modality = _MODALITY_ALIASES[_run_key]
if _selected_modality == 'all':
    SELECTED_MODALITIES = list(MODALITY_CONFIG.keys())
    OUTPUT_DIR = BASE_OUTPUT_DIR
else:
    SELECTED_MODALITIES = [_selected_modality]
    MODALITY_CONFIG = {_selected_modality: MODALITY_CONFIG[_selected_modality]}
    OUTPUT_DIR = BASE_OUTPUT_DIR / f'{_selected_modality}_only'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Models selected for this run:', SELECTED_MODALITIES)

STAGE2_OOF_PATH = STAGE2_DIR / get_value('science.tables.predictions_004', 'predictions_004.csv')
STAGE2_SUMMARY_PATH = STAGE2_DIR / get_value('science.tables.metrics_004', 'metrics_004.csv')



for model_name, cfg in MODALITY_CONFIG.items():
    print(f"{model_name}: {cfg['external_matrix']}")


In [ ]:
def normalize_sample_id(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if re.fullmatch('-?\\d+\\.0', s):
        s = s[:-2]
    return s

def find_column(df, candidates, required=True):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        key = str(candidate).strip().lower()
        if key in lower_map:
            return lower_map[key]
    if required:
        raise ValueError(f'Cannot find any of columns {candidates}. Existing columns: {list(df.columns)}')
    return None

def to_binary_response(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().lower()
        positive = {'1', 'yes', 'y', 'responder', 'response', 'r', 'cr', 'pr', 'orr'}
        negative = {'0', 'no', 'n', 'nonresponder', 'non-responder', 'nr', 'sd', 'pd'}
        if s in positive:
            return 1
        if s in negative:
            return 0
    try:
        return int(float(x))
    except Exception as exc:
        raise ValueError(f'Cannot convert Response value to binary: {x!r}') from exc

def canonicalize_cohort(x):
    s = str(x).strip().lower()
    if 'external' in s or s in {'ext', 'validation', 'test'}:
        return 'external'
    if 'internal' in s or s in {'int', 'training', 'train'}:
        return 'internal'
    return s

def canonicalize_modal(x):
    s = str(x).strip().lower().replace('-', '_').replace(' ', '_')
    if 'radio' in s or s in {'ct', 'imaging', 'image'}:
        return 'radiomics'
    if 'blood' in s or 'plasma' in s or 'serum' in s:
        return 'blood'
    if 'urine' in s:
        return 'urine'
    if any((k in s for k in ['microbi', 'metagen', 'species', 'microbiome', 'ko'])):
        return 'microbiota'
    if 'all_4' in s or 'all4' in s or 'multimodal' in s:
        return 'all_4_modal'
    return s

def read_external_label_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'External label file not found: {path}')
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        raw = pd.read_excel(path)
    else:
        raw = pd.read_csv(path, encoding='utf-8-sig')
    raw.columns = [str(c).strip() for c in raw.columns]
    sample_col = find_column(raw, ['sample_id', 'Sample ID', 'sample', 'Sample', 'ID'])
    cohort_col = find_column(raw, ['Cohort', 'cohort'])
    modal_col = find_column(raw, ['Modal', 'modal', 'Modality', 'modality'])
    response_col = find_column(raw, ['Response', 'response', 'Group', 'ORR'])
    pfs_time_col = find_column(raw, ['PFS_time', 'pfs_time', 'PFS', 'PFS_month', 'PFS_months'])
    pfs_status_col = find_column(raw, ['PFS_status', 'pfs_status', 'PFS_event', 'pfs_event'])
    os_time_col = find_column(raw, ['OS_time', 'os_time', 'OS', 'OS_month', 'OS_months'])
    os_status_col = find_column(raw, ['OS_status', 'os_status', 'OS_event', 'os_event'])
    out = pd.DataFrame({'sample': raw[sample_col].map(normalize_sample_id), 'Cohort': raw[cohort_col].map(canonicalize_cohort), 'Modal': raw[modal_col].map(canonicalize_modal), 'Response': raw[response_col].map(to_binary_response), 'PFS_time': pd.to_numeric(raw[pfs_time_col], errors='coerce'), 'PFS_status': pd.to_numeric(raw[pfs_status_col], errors='coerce'), 'OS_time': pd.to_numeric(raw[os_time_col], errors='coerce'), 'OS_status': pd.to_numeric(raw[os_status_col], errors='coerce')})
    out = out[out['Cohort'] == 'external'].copy()
    out = out.dropna(subset=['sample'])
    out['Response'] = pd.to_numeric(out['Response'], errors='coerce')
    required_outcomes = ['Response', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
    missing_summary = out[required_outcomes].isna().sum()
    if missing_summary.sum() > 0:
        print('Warning: missing external outcomes before modality-specific filtering:')
        print(missing_summary[missing_summary > 0])
    
    print('External records by modality:')
    
    return out

def read_model_ready_matrix(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Model-ready matrix not found: {path}')
    df = pd.read_csv(path, encoding='utf-8-sig')
    df.columns = [str(c).strip() for c in df.columns]
    sample_col = find_column(df, ['sample', 'sample_id', 'Sample ID', 'Sample', 'ID'], required=False)
    if sample_col is None:
        sample_col = df.columns[0]
    df = df.rename(columns={sample_col: 'sample'})
    df['sample'] = df['sample'].map(normalize_sample_id)
    df = df.dropna(subset=['sample'])
    duplicate_samples = df['sample'].duplicated(keep=False)
    if duplicate_samples.any():
        dup_values = df.loc[duplicate_samples, 'sample'].tolist()[:10]
        raise ValueError(f'Duplicated sample IDs in {path}: {dup_values}')
    df = df.set_index('sample')
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df = df.apply(pd.to_numeric, errors='coerce')
    return df
external_labels_all = read_external_label_table(EXTERNAL_LABEL_PATH)


In [ ]:
class ReducedMultiTaskNet(nn.Module):

    def __init__(self, input_dim, hidden1=64, hidden2=16, dropout=0.4):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.BatchNorm1d(hidden1), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden1, hidden2), nn.BatchNorm1d(hidden2), nn.ReLU(), nn.Dropout(dropout))
        self.response_head = nn.Linear(hidden2, 1)
        self.pfs_head = nn.Linear(hidden2, 1)
        self.os_head = nn.Linear(hidden2, 1)

    def forward(self, x):
        z = self.encoder(x)
        return {'response_logit': self.response_head(z).view(-1), 'pfs_risk': self.pfs_head(z).view(-1), 'os_risk': self.os_head(z).view(-1)}

def trusted_torch_load(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def infer_architecture_from_state_dict(state_dict, input_dim):
    try:
        hidden1 = int(state_dict['encoder.0.weight'].shape[0])
        hidden2 = int(state_dict['encoder.4.weight'].shape[0])
    except KeyError as exc:
        raise KeyError('Cannot infer ReducedMultiTaskNet hidden dimensions from checkpoint.') from exc
    return (hidden1, hidden2)

def load_frozen_model_package(model_name):
    model_dir = STAGE3_DIR / model_name
    checkpoint_path = model_dir / f"{model_name}{get_value('science.artifacts.transform_003', 'transform_003.pt')}"
    imputer_path = model_dir / f"{model_name}{get_value('science.artifacts.transform_001', 'transform_001.joblib')}"
    internal_prediction_path = model_dir / f"{model_name}{get_value('science.tables.predictions_007', 'predictions_007.csv')}"
    required = [checkpoint_path, imputer_path, internal_prediction_path]
    missing = [p for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError('Missing frozen model package files:\n' + '\n'.join((str(p) for p in missing)))
    checkpoint = trusted_torch_load(checkpoint_path, map_location=DEVICE)
    state_dict = checkpoint['model_state_dict']
    feature_names = [str(x) for x in checkpoint['feature_names']]
    input_dim = int(checkpoint['input_dim'])
    if input_dim != len(feature_names):
        raise ValueError(f'{model_name}: checkpoint input_dim={input_dim}, but feature_names={len(feature_names)}')
    hidden1, hidden2 = infer_architecture_from_state_dict(state_dict, input_dim)
    model = ReducedMultiTaskNet(input_dim=input_dim, hidden1=hidden1, hidden2=hidden2, dropout=0.0).to(DEVICE)
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    imputer = joblib.load(imputer_path)
    internal_predictions = pd.read_csv(internal_prediction_path, index_col=0, encoding='utf-8-sig')
    internal_predictions.index = internal_predictions.index.map(normalize_sample_id)
    return {'model': model, 'checkpoint': checkpoint, 'feature_names': feature_names, 'imputer': imputer, 'internal_predictions': internal_predictions, 'paths': {'checkpoint': checkpoint_path, 'imputer': imputer_path, 'internal_predictions': internal_prediction_path}}


In [ ]:
def safe_auc(y_true, y_score):
    y_true = np.asarray(y_true, dtype=float)
    y_score = np.asarray(y_score, dtype=float)
    ok = np.isfinite(y_true) & np.isfinite(y_score)
    y_true, y_score = (y_true[ok], y_score[ok])
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_score))

def safe_auprc(y_true, y_score):
    y_true = np.asarray(y_true, dtype=float)
    y_score = np.asarray(y_score, dtype=float)
    ok = np.isfinite(y_true) & np.isfinite(y_score)
    y_true, y_score = (y_true[ok], y_score[ok])
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, y_score))

def safe_cindex(time, event, risk):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=float)
    risk = np.asarray(risk, dtype=float)
    ok = np.isfinite(time) & np.isfinite(event) & np.isfinite(risk) & (time > 0)
    time, event, risk = (time[ok], event[ok], risk[ok])
    if len(time) < 3 or event.sum() < 1:
        return np.nan
    try:
        return float(concordance_index(time, -risk, event))
    except Exception:
        return np.nan

def bootstrap_ci(df, metric_function, required_cols, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    work = df[list(required_cols)].dropna().reset_index(drop=True)
    n = len(work)
    if n < 3:
        return (np.nan, np.nan, np.nan, 0)
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        sample = work.iloc[idx]
        try:
            value = metric_function(sample)
        except Exception:
            value = np.nan
        if np.isfinite(value):
            estimates.append(float(value))
    if len(estimates) == 0:
        return (np.nan, np.nan, np.nan, 0)
    estimates = np.asarray(estimates, dtype=float)
    lower = float(np.quantile(estimates, BOOTSTRAP_ALPHA / 2))
    upper = float(np.quantile(estimates, 1 - BOOTSTRAP_ALPHA / 2))
    return (float(np.mean(estimates)), lower, upper, len(estimates))

def derive_response_thresholds_from_oof(path):
    thresholds = {}
    path = Path(path)
    if not path.exists():
        print('Warning: Stage 2 OOF prediction file not found; threshold 0.5 will be used.')
        return thresholds
    oof = pd.read_csv(path, encoding='utf-8-sig')
    required = {'sample', 'model', 'Response_true', 'Response_pred'}
    if not required.issubset(oof.columns):
        print('Warning: Stage 2 OOF file lacks required columns; threshold 0.5 will be used.')
        return thresholds
    oof['sample'] = oof['sample'].map(normalize_sample_id)
    patient_oof = oof.groupby(['model', 'sample'], as_index=False).agg(Response_true=('Response_true', 'first'), Response_pred=('Response_pred', 'mean'))
    for model_name, sub in patient_oof.groupby('model'):
        y = pd.to_numeric(sub['Response_true'], errors='coerce').values
        p = pd.to_numeric(sub['Response_pred'], errors='coerce').values
        ok = np.isfinite(y) & np.isfinite(p)
        y, p = (y[ok], p[ok])
        if len(np.unique(y)) < 2:
            continue
        fpr, tpr, candidate_thresholds = roc_curve(y, p)
        finite = np.isfinite(candidate_thresholds)
        if finite.sum() == 0:
            continue
        youden = tpr[finite] - fpr[finite]
        thresholds[str(model_name)] = float(candidate_thresholds[finite][np.argmax(youden)])
    return thresholds

def classification_metrics_at_threshold(y_true, y_prob, threshold):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp) if tn + fp > 0 else np.nan
    ppv = precision_score(y_true, y_pred, zero_division=0)
    npv = tn / (tn + fn) if tn + fn > 0 else np.nan
    return {'Response_threshold': float(threshold), 'Response_balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)), 'Response_sensitivity': float(sensitivity), 'Response_specificity': float(specificity), 'Response_PPV': float(ppv), 'Response_NPV': float(npv), 'Response_F1': float(f1_score(y_true, y_pred, zero_division=0)), 'Response_TP': int(tp), 'Response_FP': int(fp), 'Response_TN': int(tn), 'Response_FN': int(fn)}

def empirical_percentile(values, reference_values, higher_is_better=True):
    values = np.asarray(values, dtype=float)
    reference = np.asarray(reference_values, dtype=float)
    reference = np.sort(reference[np.isfinite(reference)])
    if len(reference) == 0:
        return np.full(len(values), np.nan)
    pct = 100.0 * np.searchsorted(reference, values, side='right') / len(reference)
    if not higher_is_better:
        pct = 100.0 - pct
    return np.clip(pct, 0.0, 100.0)
response_thresholds = derive_response_thresholds_from_oof(STAGE2_OOF_PATH)
print('Frozen Response thresholds from aggregated internal OOF predictions:')
print(response_thresholds)


In [ ]:
def prepare_modality_external_data(model_name, config, label_table, package):
    X_all = read_model_ready_matrix(config['external_matrix'])
    labels = label_table[label_table['Modal'] == config['label_modal']].copy()
    labels = labels.drop_duplicates(subset='sample', keep='first').set_index('sample')
    feature_names = package['feature_names']
    missing_features = [f for f in feature_names if f not in X_all.columns]
    extra_features = [f for f in X_all.columns if f not in feature_names]
    if missing_features:
        message = f'{model_name}: external matrix is missing {len(missing_features)} checkpoint features. First missing features: {missing_features[:10]}'
        if STRICT_MODE:
            raise ValueError(message)
        print('Warning:', message)
        for feature in missing_features:
            X_all[feature] = np.nan
    X_required = X_all.reindex(columns=feature_names).copy()
    common_samples = X_required.index.intersection(labels.index)
    matrix_only = sorted(set(X_required.index) - set(labels.index))
    label_only = sorted(set(labels.index) - set(X_required.index))
    if len(common_samples) == 0:
        raise ValueError(f'{model_name}: no overlapping external sample IDs between matrix and labels.')
    X = X_required.loc[common_samples].copy()
    y = labels.loc[common_samples].copy()
    required_outcomes = ['Response', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
    joined = X.join(y[required_outcomes], how='inner')
    joined = joined.dropna(subset=required_outcomes)
    joined = joined[(joined['PFS_time'] > 0) & (joined['OS_time'] > 0)].copy()
    y = joined[required_outcomes].copy()
    X = joined[feature_names].copy()
    X_processed = package['imputer'].transform(X)
    if not np.isfinite(X_processed).all():
        raise ValueError(f'{model_name}: non-finite values remain after frozen imputation.')
    audit = {'model': model_name, 'display_name': config['display_name'], 'matrix_path': str(config['external_matrix']), 'label_modal': config['label_modal'], 'n_matrix_samples': int(X_all.shape[0]), 'n_label_records': int(labels.shape[0]), 'n_overlap_before_outcome_filter': int(len(common_samples)), 'n_final_external': int(X.shape[0]), 'n_checkpoint_features': int(len(feature_names)), 'n_missing_checkpoint_features': int(len(missing_features)), 'n_extra_external_features_ignored': int(len(extra_features)), 'n_matrix_samples_without_labels': int(len(matrix_only)), 'n_label_samples_without_matrix': int(len(label_only)), 'n_PFS_time_gt_OS_time': int((y['PFS_time'] > y['OS_time'] + 1e-08).sum()), 'n_OS_event_but_PFS_censored': int(((y['OS_status'] == 1) & (y['PFS_status'] == 0)).sum()), 'missing_features': '|'.join(missing_features), 'extra_features_first20': '|'.join(extra_features[:20]), 'matrix_only_first20': '|'.join(matrix_only[:20]), 'label_only_first20': '|'.join(label_only[:20])}
    return (X, X_processed, y, audit)

def predict_external(model, X_processed):
    tensor = torch.tensor(X_processed, dtype=torch.float32, device=DEVICE)
    model.eval()
    with torch.no_grad():
        output = model(tensor)
        response_pred = torch.sigmoid(output['response_logit']).cpu().numpy()
        pfs_risk = output['pfs_risk'].cpu().numpy()
        os_risk = output['os_risk'].cpu().numpy()
    return (response_pred, pfs_risk, os_risk)

def fit_frozen_risk_stratification(pred_df, endpoint, cutoff):
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    risk_col = f'{endpoint}_risk'
    group_col = f'{endpoint}_risk_group_frozen'
    work = pred_df[[time_col, status_col, risk_col]].dropna().copy()
    work = work[work[time_col] > 0].copy()
    work[group_col] = np.where(work[risk_col] >= cutoff, SURVIVAL_GROUP_LABELS[1], SURVIVAL_GROUP_LABELS[0])
    low = work[work[group_col] == SURVIVAL_GROUP_LABELS[0]]
    high = work[work[group_col] == SURVIVAL_GROUP_LABELS[1]]
    result = {'endpoint': endpoint, 'frozen_cutoff': float(cutoff), 'n_lower_risk': int(len(low)), 'n_higher_risk': int(len(high)), 'events_lower_risk': int(low[status_col].sum()), 'events_higher_risk': int(high[status_col].sum()), 'HR_higher_vs_lower': np.nan, 'HR_CI_lower': np.nan, 'HR_CI_upper': np.nan, 'Cox_P': np.nan, 'logrank_P': np.nan}
    if len(low) > 0 and len(high) > 0:
        try:
            lr = logrank_test(high[time_col], low[time_col], event_observed_A=high[status_col], event_observed_B=low[status_col])
            result['logrank_P'] = float(lr.p_value)
        except Exception:
            pass
        try:
            cox_df = work[[time_col, status_col]].copy()
            cox_df['higher_risk'] = (work[group_col] == SURVIVAL_GROUP_LABELS[1]).astype(int)
            cph = CoxPHFitter()
            cph.fit(cox_df, duration_col=time_col, event_col=status_col)
            summary = cph.summary.loc['higher_risk']
            result['HR_higher_vs_lower'] = float(summary['exp(coef)'])
            result['HR_CI_lower'] = float(summary['exp(coef) lower 95%'])
            result['HR_CI_upper'] = float(summary['exp(coef) upper 95%'])
            result['Cox_P'] = float(summary['p'])
        except Exception:
            pass
    pred_df[group_col] = np.where(pred_df[risk_col] >= cutoff, SURVIVAL_GROUP_LABELS[1], SURVIVAL_GROUP_LABELS[0])
    return (pred_df, result)

def validate_one_modality(model_name, config, label_table):
    
    print(f'>>> External transferability: {model_name}')
    
    package = load_frozen_model_package(model_name)
    X, X_processed, y, audit = prepare_modality_external_data(model_name=model_name, config=config, label_table=label_table, package=package)
    response_pred, pfs_risk, os_risk = predict_external(package['model'], X_processed)
    pred = pd.DataFrame(index=X.index)
    pred.index.name = 'sample'
    pred['model'] = model_name
    pred['external_cohort'] = config['label_modal']
    pred['Response_true'] = y['Response'].astype(int).values
    pred['Response_pred'] = response_pred
    pred['PFS_time'] = y['PFS_time'].values
    pred['PFS_status'] = y['PFS_status'].astype(int).values
    pred['PFS_risk'] = pfs_risk
    pred['OS_time'] = y['OS_time'].values
    pred['OS_status'] = y['OS_status'].astype(int).values
    pred['OS_risk'] = os_risk
    internal_pred = package['internal_predictions']
    for col in ['Response_pred', 'PFS_risk', 'OS_risk']:
        if col not in internal_pred.columns:
            raise ValueError(f'{model_name}: internal final prediction file lacks {col}.')
    pred['Response_projection_percentile'] = empirical_percentile(pred['Response_pred'], internal_pred['Response_pred'], higher_is_better=True)
    pred['PFS_benefit_percentile'] = empirical_percentile(pred['PFS_risk'], internal_pred['PFS_risk'], higher_is_better=False)
    pred['OS_benefit_percentile'] = empirical_percentile(pred['OS_risk'], internal_pred['OS_risk'], higher_is_better=False)
    pfs_cutoff = float(pd.to_numeric(internal_pred['PFS_risk'], errors='coerce').median())
    os_cutoff = float(pd.to_numeric(internal_pred['OS_risk'], errors='coerce').median())
    pred, pfs_strat = fit_frozen_risk_stratification(pred, 'PFS', pfs_cutoff)
    pred, os_strat = fit_frozen_risk_stratification(pred, 'OS', os_cutoff)
    auc = safe_auc(pred['Response_true'], pred['Response_pred'])
    auprc = safe_auprc(pred['Response_true'], pred['Response_pred'])
    brier = brier_score_loss(pred['Response_true'], pred['Response_pred'])
    pfs_c = safe_cindex(pred['PFS_time'], pred['PFS_status'], pred['PFS_risk'])
    os_c = safe_cindex(pred['OS_time'], pred['OS_status'], pred['OS_risk'])
    _, auc_lo, auc_hi, auc_valid = bootstrap_ci(pred, lambda d: safe_auc(d['Response_true'], d['Response_pred']), ['Response_true', 'Response_pred'], seed=RANDOM_STATE + 11)
    _, auprc_lo, auprc_hi, auprc_valid = bootstrap_ci(pred, lambda d: safe_auprc(d['Response_true'], d['Response_pred']), ['Response_true', 'Response_pred'], seed=RANDOM_STATE + 22)
    _, pfs_lo, pfs_hi, pfs_valid = bootstrap_ci(pred, lambda d: safe_cindex(d['PFS_time'], d['PFS_status'], d['PFS_risk']), ['PFS_time', 'PFS_status', 'PFS_risk'], seed=RANDOM_STATE + 33)
    _, os_lo, os_hi, os_valid = bootstrap_ci(pred, lambda d: safe_cindex(d['OS_time'], d['OS_status'], d['OS_risk']), ['OS_time', 'OS_status', 'OS_risk'], seed=RANDOM_STATE + 44)
    frozen_threshold = float(response_thresholds.get(model_name, 0.5))
    threshold_metrics = classification_metrics_at_threshold(pred['Response_true'], pred['Response_pred'], frozen_threshold)
    summary = {'model': model_name, 'display_name': config['display_name'], 'external_label_modal': config['label_modal'], 'n_external': int(len(pred)), 'n_response_positive': int(pred['Response_true'].sum()), 'n_pfs_events': int(pred['PFS_status'].sum()), 'n_os_events': int(pred['OS_status'].sum()), 'Response_AUROC': auc, 'Response_AUROC_CI_lower': auc_lo, 'Response_AUROC_CI_upper': auc_hi, 'Response_AUROC_bootstrap_valid': auc_valid, 'Response_AUPRC': auprc, 'Response_AUPRC_CI_lower': auprc_lo, 'Response_AUPRC_CI_upper': auprc_hi, 'Response_AUPRC_bootstrap_valid': auprc_valid, 'Response_Brier': float(brier), 'PFS_Cindex': pfs_c, 'PFS_Cindex_CI_lower': pfs_lo, 'PFS_Cindex_CI_upper': pfs_hi, 'PFS_Cindex_bootstrap_valid': pfs_valid, 'OS_Cindex': os_c, 'OS_Cindex_CI_lower': os_lo, 'OS_Cindex_CI_upper': os_hi, 'OS_Cindex_bootstrap_valid': os_valid, 'checkpoint_best_epoch': int(package['checkpoint'].get('best_epoch', -1))}
    summary.update(threshold_metrics)
    pfs_strat['model'] = model_name
    os_strat['model'] = model_name
    print(f'n={len(pred)} | Response AUROC={auc:.3f} ({auc_lo:.3f}-{auc_hi:.3f}) | AUPRC={auprc:.3f} ({auprc_lo:.3f}-{auprc_hi:.3f})')
    print(f'PFS C-index={pfs_c:.3f} ({pfs_lo:.3f}-{pfs_hi:.3f}) | OS C-index={os_c:.3f} ({os_lo:.3f}-{os_hi:.3f})')
    return {'summary': summary, 'predictions': pred, 'audit': audit, 'risk_stratification': [pfs_strat, os_strat]}


In [ ]:
all_results = {}
summary_records = []
prediction_parts = []
audit_records = []
risk_stratification_records = []
failed_modalities = []
for model_name, config in MODALITY_CONFIG.items():
    try:
        result = validate_one_modality(model_name, config, external_labels_all)
        all_results[model_name] = result
        summary_records.append(result['summary'])
        prediction_parts.append(result['predictions'].reset_index())
        audit_records.append(result['audit'])
        risk_stratification_records.extend(result['risk_stratification'])
    except Exception as exc:
        failed_modalities.append({'model': model_name, 'error': repr(exc)})
        if STRICT_MODE:
            raise
        print(f'[Skipped] {model_name}: {exc}')
if len(summary_records) == 0:
    raise RuntimeError('No external modality validation completed successfully.')
external_summary = pd.DataFrame(summary_records)
external_predictions = pd.concat(prediction_parts, ignore_index=True)
external_audit = pd.DataFrame(audit_records)
external_risk_summary = pd.DataFrame(risk_stratification_records)
failed_modalities_df = pd.DataFrame(failed_modalities)
external_summary.to_csv(OUTPUT_DIR / get_value('science.tables.metrics_022', 'metrics_022.csv'), index=False, encoding='utf-8-sig')
external_predictions.to_csv(OUTPUT_DIR / get_value('science.tables.predictions_019', 'predictions_019.csv.gz'), index=False, encoding='utf-8-sig', compression='gzip')
external_audit.to_csv(OUTPUT_DIR / get_value('science.tables.result_047', 'result_047.csv'), index=False, encoding='utf-8-sig')
external_risk_summary.to_csv(OUTPUT_DIR / get_value('science.tables.result_048', 'result_048.csv'), index=False, encoding='utf-8-sig')
if len(failed_modalities_df) > 0:
    failed_modalities_df.to_csv(OUTPUT_DIR / get_value('science.tables.result_049', 'result_049.csv'), index=False, encoding='utf-8-sig')
print('\nExternal performance summary:')

print('\nFeature/sample alignment audit:')

print('\nFrozen risk stratification summary:')



In [ ]:
RANKING_DIR = OUTPUT_DIR / get_value('science.paths.directory_051', 'directory_051')
RANKING_DIR.mkdir(parents=True, exist_ok=True)

def rank_percentile_from_rank(rank_values):
    pass
    rank_values = pd.to_numeric(pd.Series(rank_values), errors='coerce')
    n = int(rank_values.notna().sum())
    if n == 0:
        return pd.Series(np.nan, index=rank_values.index, dtype=float)
    if n == 1:
        out = pd.Series(np.nan, index=rank_values.index, dtype=float)
        out.loc[rank_values.notna()] = 100.0
        return out
    return 100.0 * (n - rank_values) / (n - 1)
ranking_parts = []
for model_name, result in all_results.items():
    pred = result['predictions'].copy()
    pred.index = pred.index.map(normalize_sample_id)
    pred.index.name = 'sample'
    rank_df = pred.reset_index().copy()
    rank_df['model'] = model_name
    rank_df['display_name'] = MODALITY_CONFIG[model_name]['display_name']
    rank_df['Response_rank'] = rank_df['Response_pred'].rank(method='min', ascending=False, na_option='bottom').astype('Int64')
    rank_df['Response_rank_percentile'] = rank_percentile_from_rank(rank_df['Response_rank'])
    rank_df['PFS_benefit_rank'] = rank_df['PFS_risk'].rank(method='min', ascending=True, na_option='bottom').astype('Int64')
    rank_df['PFS_benefit_rank_percentile'] = rank_percentile_from_rank(rank_df['PFS_benefit_rank'])
    rank_df['OS_benefit_rank'] = rank_df['OS_risk'].rank(method='min', ascending=True, na_option='bottom').astype('Int64')
    rank_df['OS_benefit_rank_percentile'] = rank_percentile_from_rank(rank_df['OS_benefit_rank'])
    output_columns = ['sample', 'model', 'display_name', 'Response_true', 'Response_pred', 'Response_rank', 'Response_rank_percentile', 'PFS_time', 'PFS_status', 'PFS_risk', 'PFS_benefit_rank', 'PFS_benefit_rank_percentile', 'OS_time', 'OS_status', 'OS_risk', 'OS_benefit_rank', 'OS_benefit_rank_percentile']
    missing_columns = [c for c in output_columns if c not in rank_df.columns]
    if missing_columns:
        raise KeyError(f'{model_name}: ranking table is missing columns {missing_columns}')
    rank_df = rank_df[output_columns].copy()
    ranking_parts.append(rank_df)
    rank_df.to_csv(RANKING_DIR / f"{model_name}{get_value('science.tables.predictions_020', 'predictions_020.csv')}", index=False, encoding='utf-8-sig')
    response_ranking = rank_df.sort_values(['Response_rank', 'sample'], ascending=[True, True])
    response_ranking.to_csv(RANKING_DIR / f"{model_name}{get_value('science.tables.predictions_021', 'predictions_021.csv')}", index=False, encoding='utf-8-sig')
    pfs_ranking = rank_df.sort_values(['PFS_benefit_rank', 'sample'], ascending=[True, True])
    pfs_ranking.to_csv(RANKING_DIR / f"{model_name}{get_value('science.tables.predictions_022', 'predictions_022.csv')}", index=False, encoding='utf-8-sig')
    os_ranking = rank_df.sort_values(['OS_benefit_rank', 'sample'], ascending=[True, True])
    os_ranking.to_csv(RANKING_DIR / f"{model_name}{get_value('science.tables.predictions_023', 'predictions_023.csv')}", index=False, encoding='utf-8-sig')
    
    print(f"{MODALITY_CONFIG[model_name]['display_name']} patient-level ranking")
    print('\nTop 20 patients ranked by predicted response:')
    
    print('\nTop 20 patients ranked by predicted PFS benefit:')
    
    print('\nTop 20 patients ranked by predicted OS benefit:')
    
external_patient_ranking = pd.concat(ranking_parts, ignore_index=True)
external_patient_ranking.to_csv(OUTPUT_DIR / get_value('science.tables.predictions_024', 'predictions_024.csv.gz'), index=False, encoding='utf-8-sig', compression='gzip')
print('\nCombined ranking table for all patients:')

print('\nDirectory containing the modality-specific ranking files:')



In [ ]:
if STAGE2_SUMMARY_PATH.exists():
    internal_summary = pd.read_csv(STAGE2_SUMMARY_PATH, encoding='utf-8-sig')
    keep_cols = ['model', 'Response_AUC_mean', 'Response_AUC_sd', 'Response_AUPRC_mean', 'Response_AUPRC_sd', 'PFS_Cindex_mean', 'PFS_Cindex_sd', 'OS_Cindex_mean', 'OS_Cindex_sd', 'best_epoch_median']
    keep_cols = [c for c in keep_cols if c in internal_summary.columns]
    internal_summary = internal_summary[keep_cols].copy()
    internal_external_comparison = internal_summary.merge(external_summary, on='model', how='inner', suffixes=('_internal', '_external'))
    internal_external_comparison.to_csv(OUTPUT_DIR / get_value('science.tables.metrics_023', 'metrics_023.csv'), index=False, encoding='utf-8-sig')
    
else:
    internal_external_comparison = pd.DataFrame()
    print('Stage 2 internal summary not found; comparison table skipped.')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
roc_legend = []
for model_name, result in all_results.items():
    pred = result['predictions']
    y = pred['Response_true'].values
    p = pred['Response_pred'].values
    if len(np.unique(y)) < 2:
        continue
    fpr, tpr, _ = roc_curve(y, p)
    auc = roc_auc_score(y, p)
    line, = ax.plot(fpr, tpr, lw=2)
    roc_legend.append((line, f"{MODALITY_CONFIG[model_name]['display_name']} | AUROC={auc:.3f}"))
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', lw=1)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('Modality-specific external Response ROC')
ax.grid(alpha=0.2)
if roc_legend:
    ax.legend([x[0] for x in roc_legend], [x[1] for x in roc_legend], frameon=False)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / get_value('science.figures.plot_066', 'plot_066.png'), bbox_inches='tight')
fig.savefig(OUTPUT_DIR / get_value('science.figures.plot_067', 'plot_067.pdf'), bbox_inches='tight')
plt.show()
fig, ax = plt.subplots(figsize=(9, 7))
pr_legend = []
for model_name, result in all_results.items():
    pred = result['predictions']
    y = pred['Response_true'].values
    p = pred['Response_pred'].values
    if len(np.unique(y)) < 2:
        continue
    precision, recall, _ = precision_recall_curve(y, p)
    auprc = average_precision_score(y, p)
    line, = ax.plot(recall, precision, lw=2)
    pr_legend.append((line, f"{MODALITY_CONFIG[model_name]['display_name']} | AUPRC={auprc:.3f}"))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Modality-specific external Response precision-recall curves')
ax.grid(alpha=0.2)
if pr_legend:
    ax.legend([x[0] for x in pr_legend], [x[1] for x in pr_legend], frameon=False)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / get_value('science.figures.plot_068', 'plot_068.png'), bbox_inches='tight')
fig.savefig(OUTPUT_DIR / get_value('science.figures.plot_069', 'plot_069.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
forest_specs = [('Response_AUROC', 'Response_AUROC_CI_lower', 'Response_AUROC_CI_upper', 'Response AUROC'), ('Response_AUPRC', 'Response_AUPRC_CI_lower', 'Response_AUPRC_CI_upper', 'Response AUPRC'), ('PFS_Cindex', 'PFS_Cindex_CI_lower', 'PFS_Cindex_CI_upper', 'PFS C-index'), ('OS_Cindex', 'OS_Cindex_CI_lower', 'OS_Cindex_CI_upper', 'OS C-index')]
for metric, lower_col, upper_col, title in forest_specs:
    plot_df = external_summary.sort_values(metric, ascending=True).reset_index(drop=True)
    y = np.arange(len(plot_df))
    values = plot_df[metric].astype(float).values
    lower = plot_df[lower_col].astype(float).values
    upper = plot_df[upper_col].astype(float).values
    xerr = np.vstack([values - lower, upper - values])
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.errorbar(values, y, xerr=xerr, fmt='o', capsize=4, lw=1.5)
    ax.axvline(0.5, linestyle='--', color='gray', lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df['display_name'])
    ax.set_xlim(0, 1)
    ax.set_xlabel(title)
    ax.set_title(f'External {title} with bootstrap 95% CI')
    ax.grid(axis='x', alpha=0.2)
    for i, (v, lo, hi) in enumerate(zip(values, lower, upper)):
        ax.text(min(0.98, hi + 0.02), i, f'{v:.3f} ({lo:.3f}–{hi:.3f})', va='center', fontsize=9)
    plt.tight_layout()
    safe_title = title.replace(' ', '_').replace('-', '')
    fig.savefig(OUTPUT_DIR / f'External_forest_{safe_title}.png', bbox_inches='tight')
    fig.savefig(OUTPUT_DIR / f'External_forest_{safe_title}.pdf', bbox_inches='tight')
    plt.show()


In [ ]:
def plot_external_km_panel(endpoint):
    n_models = len(all_results)
    n_cols = 2
    n_rows = math.ceil(n_models / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 5.5 * n_rows), squeeze=False)
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    group_col = f'{endpoint}_risk_group_frozen'
    for ax in axes.ravel():
        ax.set_visible(False)
    for i, (model_name, result) in enumerate(all_results.items()):
        ax = axes.ravel()[i]
        ax.set_visible(True)
        pred = result['predictions'].copy()
        kmf = KaplanMeierFitter()
        for group in SURVIVAL_GROUP_LABELS:
            sub = pred[pred[group_col] == group]
            if len(sub) == 0:
                continue
            kmf.fit(sub[time_col], event_observed=sub[status_col], label=f'{group} (n={len(sub)})')
            kmf.plot_survival_function(ax=ax, ci_show=True, show_censors=True, lw=1.8)
        row = external_risk_summary[(external_risk_summary['model'] == model_name) & (external_risk_summary['endpoint'] == endpoint)]
        if len(row) == 1:
            r = row.iloc[0]
            hr_text = f"HR={r['HR_higher_vs_lower']:.2f} ({r['HR_CI_lower']:.2f}–{r['HR_CI_upper']:.2f})" if np.isfinite(r['HR_higher_vs_lower']) else 'HR=NA'
            p_text = f"log-rank P={r['logrank_P']:.3g}" if np.isfinite(r['logrank_P']) else 'P=NA'
        else:
            hr_text, p_text = ('HR=NA', 'P=NA')
        ax.set_title(f"{MODALITY_CONFIG[model_name]['display_name']}\n{hr_text}; {p_text}")
        ax.set_xlabel(f'{endpoint} time')
        ax.set_ylabel('Survival probability')
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.2)
        ax.legend(frameon=False, fontsize=9)
    fig.suptitle(f'External {endpoint} stratification using frozen internal risk cutoff', fontsize=15, y=1.01)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f"External_{endpoint}{get_value('science.figures.plot_070', 'plot_070.png')}", bbox_inches='tight')
    fig.savefig(OUTPUT_DIR / f"External_{endpoint}{get_value('science.figures.plot_071', 'plot_071.pdf')}", bbox_inches='tight')
    plt.show()
plot_external_km_panel('PFS')
plot_external_km_panel('OS')


In [ ]:
if len(internal_external_comparison) > 0:
    comparison_specs = [('Response_AUC_mean', 'Response_AUROC', 'Response AUROC'), ('Response_AUPRC_mean', 'Response_AUPRC', 'Response AUPRC'), ('PFS_Cindex_mean', 'PFS_Cindex', 'PFS C-index'), ('OS_Cindex_mean', 'OS_Cindex', 'OS C-index')]
    for internal_col, external_col, title in comparison_specs:
        if internal_col not in internal_external_comparison.columns:
            continue
        plot_df = internal_external_comparison.copy()
        x = np.arange(len(plot_df))
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.scatter(x - 0.1, plot_df[internal_col], s=65, label='Internal repeated-CV mean')
        ax.scatter(x + 0.1, plot_df[external_col], s=65, label='External independent test')
        for i, row in plot_df.iterrows():
            ax.plot([x[i] - 0.1, x[i] + 0.1], [row[internal_col], row[external_col]], lw=1, alpha=0.6)
        ax.axhline(0.5, linestyle='--', color='gray', lw=1)
        ax.set_xticks(x)
        ax.set_xticklabels([MODALITY_CONFIG.get(m, {}).get('display_name', m) for m in plot_df['model']], rotation=15, ha='right')
        ax.set_ylim(0, 1)
        ax.set_ylabel(title)
        ax.set_title(f'Internal repeated-CV versus external test: {title}')
        ax.grid(axis='y', alpha=0.2)
        ax.legend(frameon=False)
        plt.tight_layout()
        safe_title = title.replace(' ', '_').replace('-', '')
        fig.savefig(OUTPUT_DIR / f'Internal_vs_external_{safe_title}.png', bbox_inches='tight')
        fig.savefig(OUTPUT_DIR / f'Internal_vs_external_{safe_title}.pdf', bbox_inches='tight')
        plt.show()
else:
    print('Internal-versus-external plots skipped because Stage 2 summary was unavailable.')


In [ ]:
locked_cols = ['model', 'display_name', 'n_external', 'Response_AUROC', 'Response_AUROC_CI_lower', 'Response_AUROC_CI_upper', 'Response_AUPRC', 'Response_AUPRC_CI_lower', 'Response_AUPRC_CI_upper', 'PFS_Cindex', 'PFS_Cindex_CI_lower', 'PFS_Cindex_CI_upper', 'OS_Cindex', 'OS_Cindex_CI_lower', 'OS_Cindex_CI_upper']
locked_result_table = external_summary[locked_cols].copy()
locked_result_table.to_csv(OUTPUT_DIR / get_value('science.tables.metrics_024', 'metrics_024.csv'), index=False, encoding='utf-8-sig')
print('External locked performance:')

print('\nPatient ranking preview:')


for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        pass
